In [1]:
import joblib
import pandas as pd
import numpy as np

In [2]:
loaded_xgb = joblib.load("../models/xgboost_rain_model.pkl")
scaler = joblib.load("../models/scaler.pkl")
feature_names = loaded_xgb.get_booster().feature_names

print("✅ Model and scaler downloaded.")
print(f"Feature number: {len(feature_names)}")

✅ Model and scaler downloaded.
Feature number: 116


In [3]:
# Ortak değerler
common = {
    'MinTemp': 14.0,
    'MaxTemp': 28.0,
    'Rainfall': 0.0,
    'Evaporation': 8.0,
    'WindGustSpeed': 25.0,
    'WindSpeed9am': 10.0,
    'WindSpeed3pm': 12.0,
    'Humidity9am': 30.0,
    'Pressure9am': 1025.0,
    'Cloud9am': 0.0,
    'Temp9am': 18.0,
    'Temp3pm': 24.0,
    'Month_Sin': np.sin(2 * np.pi * 1 / 12.0),
    'Month_Cos': np.cos(2 * np.pi * 1 / 12.0),
    'RainToday': 0.0
}

# ☀️ Sunny
sunny = common.copy()
sunny.update({
    'Humidity3pm': 15.0,
    'Pressure3pm': 1030.0,
    'Sunshine': 12.0,
    'Cloud3pm': 0.0,
    'Temp_Range': 28.0 - 14.0,
    'Humidity_Temp_Interaction': 15.0 * 28.0,
    'Pressure_Drop': 1025.0 - 1030.0
})

# ⛈️ Stormy
stormy = common.copy()
stormy.update({
    'MaxTemp': 16.0,
    'Humidity3pm': 95.0,
    'Pressure3pm': 990.0,
    'Sunshine': 0.0,
    'Cloud3pm': 8.0,
    'Rainfall': 25.0,
    'Humidity9am': 90.0,
    'Pressure9am': 995.0,
    'Cloud9am': 8.0,
    'Temp_Range': 16.0 - 12.0,
    'Humidity_Temp_Interaction': 95.0 * 16.0,
    'Pressure_Drop': 995.0 - 990.0,
    'RainToday': 1.0
})

# 🌥️ Edge
edge = common.copy()
edge.update({
    'MaxTemp': 22.0,
    'MinTemp': 14.0,
    'Humidity3pm': 65.0,
    'Pressure3pm': 1012.0,
    'Sunshine': 4.0,
    'Cloud3pm': 4.0,
    'Temp_Range': 22.0 - 14.0,
    'Humidity_Temp_Interaction': 65.0 * 22.0,
    'Pressure_Drop': 1025.0 - 1012.0
})

print("✅ 3 senaryo verisi hazırlandı.")

✅ 3 senaryo verisi hazırlandı.


In [4]:
def build_scenario(data_dict, label):
    df = pd.DataFrame(np.zeros((1, len(feature_names))), columns=feature_names)

    # Sayısal ve türemiş kolonları doldur
    for col, val in data_dict.items():
        if col in df.columns:
            df[col] = val

    # One‑hot kolonlar (sabit seçim: Sydney, W, SW, W)
    for col in ['Location_Sydney', 'WindGustDir_W', 'WindDir9am_SW', 'WindDir3pm_W']:
        if col in df.columns:
            df[col] = 1.0

    # Sıralamayı garantile
    df = df[feature_names]
    return df


df_sunny = build_scenario(sunny, "Sunny")
df_stormy = build_scenario(stormy, "Stormy")
df_edge = build_scenario(edge, "Edge")

print("✅ 3 DataFrame oluşturuldu.")

✅ 3 DataFrame oluşturuldu.


In [5]:
def build_scenario(data_dict):
    df = pd.DataFrame(np.zeros((1, len(feature_names))), columns=feature_names)
    for col, val in data_dict.items():
        if col in df.columns:
            df[col] = val

    # One‑hot kolonlar
    for col in ['Location_Sydney', 'WindGustDir_W', 'WindDir9am_SW', 'WindDir3pm_W']:
        if col in df.columns:
            df[col] = 1.0

    return df[feature_names]


df_sunny = build_scenario(sunny)
df_stormy = build_scenario(stormy)
df_edge = build_scenario(edge)

print("✅ 3 DataFrame oluşturuldu.")

✅ 3 DataFrame oluşturuldu.


In [6]:
scaler_features = [f for f in feature_names if f != 'RainToday']

dfs = {
    '☀️ Sunny': df_sunny,
    '⛈️ Stormy': df_stormy,
    '🌥️ Edge': df_edge
}

for name, df in dfs.items():
    df_scaled = df.copy()
    df_scaled[scaler_features] = scaler.transform(df[scaler_features])

    prob = loaded_xgb.predict_proba(df_scaled)[0][1] * 100
    pred = loaded_xgb.predict(df_scaled)[0]

    print(f"{name}: Probability = %{prob:.2f}  →  Desicion = {'YES' if pred == 1 else 'NO'}")

☀️ Sunny: Probability = %0.31  →  Desicion = NO
⛈️ Stormy: Probability = %98.51  →  Desicion = YES
🌥️ Edge: Probability = %35.96  →  Desicion = NO
